In [5]:
# ============================================================
#  Cell 0 — Download dataset from Kaggle via kagglehub
# ============================================================
import kagglehub
harshar27_10kapi_path = kagglehub.dataset_download('harshar27/10kapi')
print('Data source import complete.')
print(f'Dataset path: {harshar27_10kapi_path}')


100%|██████████| 1.30M/1.30M [00:01<00:00, 850kB/s]

Extracting files...
Data source import complete.
Dataset path: /root/.cache/kagglehub/datasets/harshar27/10kapi/versions/1


In [6]:
# ============================================================
#  Cell 1 — Locate the dataset JSON file
# ============================================================
import os, glob

# Search for the JSON file in the kagglehub download path
candidates = (
    glob.glob(os.path.join(harshar27_10kapi_path, '**', '*.json'), recursive=True)
    + ['/content/api_vulnerability_dataset_10k.json']  # fallback if manually uploaded
)
DATASET_PATH = None
for p in candidates:
    if os.path.isfile(p):
        DATASET_PATH = p
        break

if DATASET_PATH:
    print(f'Found dataset: {DATASET_PATH}')
else:
    raise FileNotFoundError(
        'Could not find the dataset JSON. '
        'Upload api_vulnerability_dataset_10k.json to /content/ or check the kagglehub path.'
    )


Found dataset: /root/.cache/kagglehub/datasets/harshar27/10kapi/versions/1/api_vulnerability_dataset_10k.json


In [7]:
# ============================================================
#  Cell 2 — Quick data inspection
# ============================================================
import json

with open(DATASET_PATH) as f:
    data = json.load(f)

print('Total records:', len(data))
print('Columns:', list(data[0].keys()))
print('\nSample record:')
for k, v in data[0].items():
    print(f'  {k!r}: {str(v)[:120]}')


Total records: 10000
Columns: ['id', 'code', 'label', 'language', 'framework', 'resource', 'endpoint_path', 'flaws', 'cwe', 'severity', 'vulnerability_description', 'secure_version', 'source_dataset']

Sample record:
  'id': php_laravel_employee_00000
  'code': Route::get('/api/employee_items', function (Request $request) {
    $label = $request->input('label');
    return respon
  'label': GET
  'language': PHP
  'framework': Laravel
  'resource': employee
  'endpoint_path': /api/employee_items
  'flaws': ['sql_injection']
  'cwe': ['CWE-89']
  'severity': critical
  'vulnerability_description': SQL injection in Laravel route with raw DB::select.
  'secure_version': Route::get('/api/employee_items', function (Request $request) {
    $label = $request->input('label');
    return respon
  'source_dataset': synthetic_v2


In [8]:
# ============================================================
#  Cell 3 — Install pinned dependencies
#  IMPORTANT: After this cell, go to Runtime → Restart session
#             then run all cells FROM Cell 4 onward.
# ============================================================
!pip uninstall -y liger-kernel liger_kernel 2>/dev/null
!pip install -q \
    transformers==4.47.0 \
    trl==0.13.0 \
    peft==0.14.0 \
    bitsandbytes==0.46.0 \
    accelerate==1.2.1 \
    datasets==3.2.0 \
    triton


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 148.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 126.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 require

In [9]:
# ============================================================
#  Cell 4 — Verify package versions (run after restart)
# ============================================================
import torch, transformers, trl, peft
print(f'torch         : {torch.__version__}  (CUDA {torch.version.cuda})')
print(f'transformers  : {transformers.__version__}')
print(f'trl           : {trl.__version__}')
print(f'peft          : {peft.__version__}')

# Sanity check — these should NOT throw errors
from transformers import AutoTokenizer, AutoModelForCausalLM
print('\n✓ All imports OK')


torch         : 2.10.0+cu128  (CUDA 12.8)
transformers  : 4.47.0
trl           : 0.13.0
peft          : 0.14.0

✓ All imports OK


In [10]:
# ============================================================
#  Cell 5 — Re-locate dataset after runtime restart
# ============================================================
import os, glob, kagglehub

harshar27_10kapi_path = kagglehub.dataset_download('harshar27/10kapi')

candidates = (
    glob.glob(os.path.join(harshar27_10kapi_path, '**', '*.json'), recursive=True)
    + ['/content/api_vulnerability_dataset_10k.json']
)
DATASET_PATH = None
for p in candidates:
    if os.path.isfile(p):
        DATASET_PATH = p
        break

print(f'Dataset: {DATASET_PATH}')


Using Colab cache for faster access to the '10kapi' dataset.
Dataset: /kaggle/input/10kapi/api_vulnerability_dataset_10k.json


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# ============================================================
#  API Vulnerability Classifier — QLoRA Fine-Tuning (Colab)
#  GPU: T4 (16 GB) | Tested on Colab free tier
# ============================================================

import os
import gc
import json
import math
import warnings
from dataclasses import dataclass, field
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Colab paths ──────────────────────────────────────────────
COLAB_WORKING = Path("/content")

# ── Env flags (must be set before torch import) ──────────────
os.environ["CUDA_VISIBLE_DEVICES"]        = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]     = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]      = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import torch
import numpy as np
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig


@dataclass
class Config:
    model_name: str = "codellama/CodeLlama-7b-instruct-hf"
    dataset_json: str = DATASET_PATH
    train_split:  float = 0.85
    text_field:   str = "text"
    max_seq_len: int = 512
    load_in_4bit:   bool = True
    bnb_quant_type: str  = "nf4"
    lora_r:       int   = 16
    lora_alpha:   int   = 32
    lora_dropout: float = 0.05
    lora_target_modules: list = field(
        default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"]
    )
    epochs:             int   = 1
    batch_size:         int   = 1
    grad_accumulation:  int   = 16
    learning_rate:      float = 1e-4
    warmup_ratio:       float = 0.05
    weight_decay:       float = 0.01
    lr_scheduler:       str   = "cosine"
    max_grad_norm:      float = 0.3
    fp16: bool = True
    bf16: bool = False
    eval_steps:               int = 100
    save_steps:               int = 100
    logging_steps:            int = 10
    early_stopping_patience:  int = 3
    output_dir: str = str(COLAB_WORKING / "qlora_vuln_model")
    seed: int = 42


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def check_gpu():
    print("=" * 55)
    print("  GPU Check")
    print("=" * 55)
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No CUDA GPU found. In Colab: Runtime → Change runtime type → T4 GPU"
        )
    name = torch.cuda.get_device_name(0)
    cap  = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16 = torch.cuda.is_bf16_supported()
    print(f"  GPU   : {name}")
    print(f"  VRAM  : {vram:.1f} GB")
    print(f"  Cap   : {cap}")
    print(f"  bf16  : {'✓' if bf16 else '✗  (use fp16)'}")
    print("=" * 55)
    if "T4" in name:
        print("  ℹ  T4 detected — using fp16 (bf16 is slow on T4)")
        return False
    return bf16


def analyze_dataset(data, cfg: Config):
    print("\n[Dataset Stats]")
    texts = data["train"][cfg.text_field]
    word_counts = [len(t.split()) for t in texts]
    p50  = int(np.percentile(word_counts, 50))
    p95  = int(np.percentile(word_counts, 95))
    p99  = int(np.percentile(word_counts, 99))
    maxi = max(word_counts)
    print(f"  Train samples : {len(texts)}")
    print(f"  Val   samples : {len(data['validation'])}")
    print(f"  Words  p50/p95/p99/max : {p50}/{p95}/{p99}/{maxi}")
    if p95 > cfg.max_seq_len:
        print(f"  ⚠  p95 words ({p95}) > max_seq_len ({cfg.max_seq_len}) "
              f"— consider increasing max_seq_len")
    else:
        print(f"  ✓  max_seq_len={cfg.max_seq_len} covers p95")


# ── DATA LOADING ─────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a security-focused code reviewer specializing in API vulnerability "
    "detection and remediation. Analyze the provided code, identify security flaws, "
    "explain the vulnerabilities, and provide a secure version."
)

def format_row(row) -> dict:
    flaws = row["flaws"]
    if isinstance(flaws, list):
        flaws = ", ".join(flaws)
    cwes = row["cwe"]
    if isinstance(cwes, list):
        cwes = ", ".join(cwes)

    instruction = (
        f"Analyze the following {row['language']} ({row['framework']}) API endpoint "
        f"for security vulnerabilities.\n\n"
        f"HTTP Method : {row['label']}\n"
        f"Endpoint    : {row['endpoint_path']}\n\n"
        f"```{row['language'].lower()}\n{row['code']}\n```"
    )
    response = (
        f"## Vulnerability Analysis\n\n"
        f"**Severity**       : {row['severity'].upper()}\n"
        f"**Flaw(s)**        : {flaws}\n"
        f"**CWE**            : {cwes}\n\n"
        f"**Description**\n{row['vulnerability_description']}\n\n"
        f"## Secure Version\n\n"
        f"```{row['language'].lower()}\n{row['secure_version']}\n```"
    )
    text = (
        f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"{instruction} [/INST]\n"
        f"{response} </s>"
    )
    return {"text": text}


def load_and_split(cfg: Config) -> DatasetDict:
    print("\n[1/6] Loading & formatting dataset...")
    from datasets import Dataset

    with open(cfg.dataset_json) as f:
        records = json.load(f)

    raw = Dataset.from_list(records)
    print(f"  Raw columns : {raw.column_names}")
    print(f"  Total rows  : {len(raw)}")

    formatted = raw.map(
        format_row,
        remove_columns=raw.column_names,
        desc="Formatting prompts",
    )

    print("\n[Sample prompt — first 400 chars]")
    print("-" * 60)
    print(formatted[0]["text"][:400])
    print("-" * 60)

    split = formatted.train_test_split(
        test_size=round(1 - cfg.train_split, 2),
        seed=cfg.seed,
    )
    return DatasetDict({"train": split["train"], "validation": split["test"]})


# ── PRE-TOKENIZE ─────────────────────────────────────────────

def tokenize_dataset(dataset: DatasetDict, tokenizer, cfg: Config) -> DatasetDict:
    """Pre-tokenize so SFTTrainer receives input_ids, not raw strings."""
    print("\n[Tokenizing dataset...]")

    def tokenize_fn(examples):
        out = tokenizer(
            examples[cfg.text_field],
            truncation=True,
            max_length=cfg.max_seq_len,
            padding=False,
        )
        out["labels"] = out["input_ids"].copy()
        return out

    tokenized = dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=[cfg.text_field],
        desc="Tokenizing",
    )
    return tokenized


# ── MODEL + TOKENIZER ────────────────────────────────────────

def load_tokenizer(cfg: Config):
    print(f"\n[2/6] Loading tokenizer: {cfg.model_name}")
    tok = AutoTokenizer.from_pretrained(
        cfg.model_name,
        trust_remote_code=True,
        use_fast=False,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok


def load_model_qlora(cfg: Config, bf16_supported: bool):
    print("\n[3/6] Loading 4-bit QLoRA model...")
    cleanup()
    compute_dtype = torch.bfloat16 if bf16_supported else torch.float16

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_type=cfg.bnb_quant_type,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        quantization_config=bnb_cfg,
        device_map={"": 0},
        max_memory={0: "14GiB", "cpu": "30GiB"},
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        torch_dtype=compute_dtype,
        attn_implementation="eager",
    )

    model.config.use_cache      = False
    model.config.pretraining_tp = 1
    model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.lora_target_modules,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model


# ── SFT CONFIG ───────────────────────────────────────────────

def build_sft_config(cfg: Config, bf16_supported: bool) -> SFTConfig:
    print("\n[4/6] Building SFT config...")
    os.makedirs(cfg.output_dir, exist_ok=True)

    return SFTConfig(
        output_dir=cfg.output_dir,
        # NO dataset_text_field — we pre-tokenized
        max_seq_length=cfg.max_seq_len,
        packing=False,

        num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accumulation,

        learning_rate=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type=cfg.lr_scheduler,
        max_grad_norm=cfg.max_grad_norm,
        optim="paged_adamw_8bit",

        fp16=cfg.fp16 and not bf16_supported,
        bf16=cfg.bf16 and bf16_supported,

        eval_strategy="steps",
        eval_steps=cfg.eval_steps,
        save_strategy="steps",
        save_steps=cfg.save_steps,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        logging_steps=cfg.logging_steps,
        report_to="none",

        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
    )


# ── TRAINING ─────────────────────────────────────────────────

def train(cfg: Config):
    cleanup()
    bf16_ok = check_gpu()

    dataset  = load_and_split(cfg)
    analyze_dataset(dataset, cfg)

    tokenizer = load_tokenizer(cfg)

    # Pre-tokenize BEFORE passing to SFTTrainer
    tok_dataset = tokenize_dataset(dataset, tokenizer, cfg)

    model     = load_model_qlora(cfg, bf16_ok)
    sft_cfg   = build_sft_config(cfg, bf16_ok)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    print("\n[5/6] Initialising SFTTrainer...")
    trainer = SFTTrainer(
        model=model,
        args=sft_cfg,
        train_dataset=tok_dataset["train"],
        eval_dataset=tok_dataset["validation"],
        data_collator=data_collator,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=cfg.early_stopping_patience
            )
        ],
    )

    print("\n[6/6] Training...")
    trainer.train()

    final = os.path.join(cfg.output_dir, "final_adapter")
    model.save_pretrained('/content/drive/my_model')
    tokenizer.save_pretrained('/content/drive/my_model')
    trainer.save_model(final)
    tokenizer.save_pretrained(final)
    print(f"\n✓ Training complete.  Adapter saved → {final}")

    return trainer


if __name__ == "__main__":
    set_seed(42)
    cfg = Config()
    trainer = train(cfg)

  GPU Check
  GPU   : Tesla T4
  VRAM  : 15.6 GB
  Cap   : (7, 5)
  bf16  : ✓
  ℹ  T4 detected — using fp16 (bf16 is slow on T4)

[1/6] Loading & formatting dataset...
  Raw columns : ['id', 'code', 'label', 'language', 'framework', 'resource', 'endpoint_path', 'flaws', 'cwe', 'severity', 'vulnerability_description', 'secure_version', 'source_dataset']
  Total rows  : 10000


Formatting prompts:   0%|          | 0/10000 [00:00<?, ? examples/s]


[Sample prompt — first 400 chars]
------------------------------------------------------------
<s>[INST] <<SYS>>
You are a security-focused code reviewer specializing in API vulnerability detection and remediation. Analyze the provided code, identify security flaws, explain the vulnerabilities, and provide a secure version.
<</SYS>>

Analyze the following PHP (Laravel) API endpoint for security vulnerabilities.

HTTP Method : GET
Endpoint    : /api/employee_items

```php
Route::get('/api/em
------------------------------------------------------------

[Dataset Stats]
  Train samples : 8500
  Val   samples : 1500
  Words  p50/p95/p99/max : 124/169/184/193
  ✓  max_seq_len=512 covers p95

[2/6] Loading tokenizer: codellama/CodeLlama-7b-instruct-hf


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

trainable params: 16,777,216 || all params: 6,755,323,904 || trainable%: 0.2484

[4/6] Building SFT config...

[5/6] Initialising SFTTrainer...

[6/6] Training...


Step,Training Loss,Validation Loss
100,1.630600,0.094134
200,0.781600,0.049124
300,0.680000,0.043064
400,0.659500,0.041019
500,0.633500,0.040135


OSError: [Errno 95] Operation not supported: '/content/drive/my_model'

In [3]:
!pip install -U bitsandbytes>=0.46.1

In [7]:

# FIX: Upgrade bitsandbytes to the required version before imports
!pip install -U bitsandbytes>=0.46.1


"""
Inference test for the fine-tuned QLoRA adapter.
Run in Colab or any environment with a GPU.

Requirements:
    pip install torch transformers peft bitsandbytes accelerate
"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ── Config ────────────────────────────────────────────────────────────────────
BASE_MODEL   = "codellama/CodeLlama-7b-instruct-hf"
ADAPTER_PATH = "./content"   # adjust path as needed

SYSTEM_PROMPT = (
    "You are a security-focused code reviewer specializing in API vulnerability "
    "detection and remediation. Analyze the provided code, identify security flaws, "
    "explain the vulnerabilities, and provide a secure version."
)

test_prompt = (
    "<s>[INST] <<SYS>>\n"
    + SYSTEM_PROMPT + "\n<</SYS>>\n\n"
    "Analyze the following PHP (Laravel) API endpoint for security vulnerabilities.\n\n"
    "HTTP Method : GET\n"
    "Endpoint    : /api/users\n\n"
    "```php\n"
    "Route::get('/api/users', function(Request $r) {\n"
    "    $id = $r->input('id');\n"
    '    return DB::select("SELECT * FROM users WHERE id = $id");\n'
    "});\n"
    "``` [/INST]\n"
)

# ── Load tokenizer ─────────────────────────────────────────────────────────────
print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
tok.pad_token = tok.eos_token

# ── Load base model in 4-bit ───────────────────────────────────────────────────
print("Loading base model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# ── Attach QLoRA adapter ───────────────────────────────────────────────────────
print(f"Loading adapter from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# ── Inference ─────────────────────────────────────────────────────────────────
print("Running inference...\n" + "=" * 60)
inputs = tok(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.3,
        repetition_penalty=1.3,
        pad_token_id=tok.eos_token_id,
    )

# Decode only the newly generated tokens (skip the prompt)
generated = out[0][inputs["input_ids"].shape[1]:]
print(tok.decode(generated, skip_special_tokens=True))


Loading tokenizer...
Loading base model in 4-bit...


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [16]:
import shutil
from google.colab import files

# Compress the folder (name of zip, format, folder to zip)
shutil.make_archive('model_folder', 'zip', '/content/qlora_vuln_model')

# Download to your local machine
files.download('model_folder.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
!cp -r /content/qlora_vuln_model /content/drive/Model


cp: cannot create directory '/content/drive/Model': Operation not supported
